<!-- NB_DOC_INTRO_V1 -->
# Time Series — Vue panoramique des methodes (stat, ML, DL)

> 📚 **Doc thematique** : [docs/06_TS_TDS.md](docs/06_TS_TDS.md) (Time Series & Traitement du Signal)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Tour d'horizon** des familles de methodes de forecasting TS : modeles **statistiques** (ARIMA, ETS, Theta), **ML** avec lag features (RF, GBM), **DL** (LSTM, TCN, Transformer, N-BEATS), **specialises** (Prophet, NeuralProphet, darts, statsforecast).

Bench compact sur Air Passengers. Pour ARIMA detaille : [TS_ARIMAs_Revoir.ipynb](./TS_ARIMAs_Revoir.ipynb).

## Plan

1. Setup + dataset
2. Naive baselines (naive, seasonal-naive, mean)
3. ETS (Holt-Winters)
4. ARIMA / SARIMA
5. Prophet
6. ML : RF + lag features
7. DL : LSTM (concept + code)
8. Bench compare (MASE, sMAPE)
9. Top librairies 2024-2025 (darts, statsforecast)
10. Pieges + References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Dataset

In [ ]:
# Air Passengers synthese
dates = pd.date_range("1949-01-01", periods=144, freq="MS")
trend = np.linspace(100, 600, 144)
season = 80 * np.sin(2 * np.pi * np.arange(144) / 12)
noise = np.random.normal(0, 20, 144)
ts = pd.Series(trend + season + noise, index=dates)

# Split : 80% train, 20% test
split = int(len(ts) * 0.8)
train, test = ts[:split], ts[split:]
horizon = len(test)
print(f"Train : {len(train)}, Test : {len(test)} (horizon = {horizon} mois)")

## 2. Naive baselines

In [ ]:
# 1. Naive : repete derniere valeur
naive = pd.Series([train.iloc[-1]] * horizon, index=test.index)

# 2. Seasonal-naive : repete les 12 dernieres valeurs (saisonnalite 12)
seasonal_naive = pd.Series(np.tile(train[-12:].values, int(horizon / 12) + 1)[:horizon], index=test.index)

# 3. Mean (moyenne historique)
mean_baseline = pd.Series([train.mean()] * horizon, index=test.index)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train.index, train.values, label="Train", color="blue")
ax.plot(test.index, test.values, label="Test (vrai)", color="black", linewidth=2)
ax.plot(naive.index, naive.values, label="Naive", linestyle="--")
ax.plot(seasonal_naive.index, seasonal_naive.values, label="Seasonal-naive", linestyle="--")
ax.plot(mean_baseline.index, mean_baseline.values, label="Mean", linestyle=":")
ax.legend(); ax.set_title("Baselines")
plt.show()

## 3. ETS (Holt-Winters)

Lissage exponentiel triple (level + trend + seasonality). Tres bon baseline en forecasting.

In [ ]:
ets = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12,
                            initialization_method="estimated").fit()
ets_forecast = ets.forecast(horizon)
print(f"ETS — RMSE : {root_mean_squared_error(test, ets_forecast):.2f}")

## 4. ARIMA / SARIMA

In [ ]:
# SARIMA(1,1,1)(1,1,1)[12] : differencie + AR(1) + MA(1) + saisonnier
sarima = ARIMA(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)).fit()
sarima_forecast = sarima.forecast(horizon)
print(f"SARIMA — RMSE : {root_mean_squared_error(test, sarima_forecast):.2f}")

## 5. Prophet (Meta — robuste, peu de tuning)

In [ ]:
try:
    from prophet import Prophet
    df_p = pd.DataFrame({"ds": train.index, "y": train.values})
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m.fit(df_p)
    future = pd.DataFrame({"ds": test.index})
    prophet_forecast = m.predict(future)["yhat"].values
    print(f"Prophet — RMSE : {root_mean_squared_error(test, prophet_forecast):.2f}")
    PROPHET_OK = True
except ImportError:
    print("Prophet pas installe : uv add --group ts prophet")
    prophet_forecast = None
    PROPHET_OK = False

## 6. ML — Random Forest avec lag features

In [ ]:
def make_lag_features(ts, lags=[1, 2, 3, 6, 12]):
    df = pd.DataFrame({"y": ts})
    for lag in lags:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    df["month"] = df.index.month
    return df.dropna()

ts_feat = make_lag_features(ts)
train_feat = ts_feat[:split].dropna()
test_feat = ts_feat[split:]

X_tr, y_tr = train_feat.drop(columns="y"), train_feat["y"]
X_te, y_te = test_feat.drop(columns="y"), test_feat["y"]

rf = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)
rf_forecast = pd.Series(rf.predict(X_te), index=test_feat.index)
print(f"RF lag features — RMSE : {root_mean_squared_error(y_te, rf_forecast):.2f}")

## 7. DL — LSTM (concept + code de reference)

Les LSTM/GRU/Transformer dominent sur les TS multivariees longues. Code pret a executer :

In [ ]:
# Pseudo-code minimaliste LSTM en PyTorch
example_code = '''
import torch, torch.nn as nn

class LSTMForecaster(nn.Module):
    def __init__(self, input_size=1, hidden=64, n_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):    # x shape (batch, seq_len, input_size)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])      # garde dernier timestep

# Preparation : transformer la TS en (X, y) sliding windows (voir TS_Generer_Sequence.ipynb)
# Train avec MSELoss + Adam, batch_size=32, epochs=50-100
'''
print(example_code)

## 8. Comparatif final

In [ ]:
results = pd.DataFrame({
    "model": ["Naive", "Seasonal-Naive", "ETS", "SARIMA", "RF lags"],
    "RMSE":  [
        root_mean_squared_error(test, naive),
        root_mean_squared_error(test, seasonal_naive),
        root_mean_squared_error(test, ets_forecast),
        root_mean_squared_error(test, sarima_forecast),
        root_mean_squared_error(y_te, rf_forecast),
    ],
    "MAE":  [
        mean_absolute_error(test, naive),
        mean_absolute_error(test, seasonal_naive),
        mean_absolute_error(test, ets_forecast),
        mean_absolute_error(test, sarima_forecast),
        mean_absolute_error(y_te, rf_forecast),
    ],
})
if PROPHET_OK:
    results = pd.concat([results, pd.DataFrame({
        "model": ["Prophet"],
        "RMSE": [root_mean_squared_error(test, prophet_forecast)],
        "MAE":  [mean_absolute_error(test, prophet_forecast)],
    })], ignore_index=True)
print(results.sort_values("RMSE").round(2))

## 9. Top librairies 2024-2025

| Lib | Forces |
|---|---|
| **`statsmodels`** | ARIMA, SARIMA, ETS, VAR (foundation) |
| **`statsforecast`** (Nixtla) | ARIMA / ETS ultra-rapide (Numba), batch sur plusieurs TS |
| **`pmdarima`** | Auto-ARIMA (auto-selection p, d, q) |
| **`prophet`** (Meta) | Robuste, peu de tuning, holidays |
| **`neuralprophet`** | Version NN de Prophet |
| **`darts`** | Unifie tous les modeles TS (stat + ML + DL) |
| **`sktime`** | sklearn-like API pour TS |
| **`tsai`** | DL pour TS (TimeSeriesAI) |
| **`gluonts`** (AWS) | Deep TS production-ready |

## 10. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Ne pas tester les baselines (naive, seasonal-naive) | Toujours start avec baselines simples |
| Comparer modeles sur MSE seul | MASE plus interpretable (relatif au naive) |
| Cross-validation random | TimeSeriesSplit ou walk-forward |
| Train sur dataset entier, test sur le meme | Time-based split obligatoire |
| Lag features sans `shift(1)` dans rolling | Leak |
| Pas de feature saisonniere | Sin/cos cyclic features tres utiles |
| Choisir un seul horizon | Tester multi-step (recursive vs direct) |
| Confondre forecast et nowcasting | Forecast = futur, nowcasting = present incertain |


## References

### Documentation
- Hyndman *Forecasting: Principles and Practice* : https://otexts.com/fpp3/
- Nixtla statsforecast : https://nixtla.github.io/statsforecast/
- darts : https://unit8co.github.io/darts/
- Prophet : https://facebook.github.io/prophet/

### Voir aussi
- [TS_Time_Series_Intro.ipynb](TS_Time_Series_Intro.ipynb)
- [TS_ARIMAs_Revoir.ipynb](TS_ARIMAs_Revoir.ipynb)
- [TS_Generer_Sequence.ipynb](TS_Generer_Sequence.ipynb)
- [TS_Maintenance_Predictive_GOOD.ipynb](TS_Maintenance_Predictive_GOOD.ipynb)
- [DS_Metrics_Reference.ipynb](DS_Metrics_Reference.ipynb)
